# Lab 2: Build a handwritten-digit recogniser — guided version
**Contemporary Machine Learning in Theory and Practice · WS26/27**

**Tutorial:** 5 November 2026 · 90 minutes  
**Submission:** follow the deadline and upload instructions on Moodle · Groups of at most four  
**Group name:**  
**Group members:**

We will build a convolutional neural network (CNN), train it on handwritten digits,
and investigate its predictions. Our dataset is **MNIST**: grayscale images of
digits with labels from 0 to 9. We build on the tensors and regression example
from Lab 1 and the image architectures from Lecture 2.

Choose **one** version of this lab and submit **one completed notebook**, including
tutorial and homework answers. Both versions have the same project, exercise
numbers, and core requirements. You may consult the guided version for hints
even if you work independently. The independent version gives you more freedom
over the architecture and asks you to write more code; it is not an additional submission.

This version breaks the project into smaller programming tasks and offers examples and hints. You write complete short pieces of code, choose an architecture for homework, and explain your observations.

| Exercise | Where to work | Suggested time |
|---|---|---:|
| 1. Explore the digit images | Tutorial | 10 min |
| 2. Construct a CNN | Tutorial | 25 min |
| 3. Train the classifier | Tutorial | 20 min |
| 4. Evaluate and inspect predictions | Tutorial | 20 min |
| 5. Design a second architecture | Homework | Work after class |
| 6. Recognise new digits | Tutorial | 10 min |
| 7. Test stability under random noise | Homework | Work after class |
| Questions and discussion | Tutorial | 5 min |

**In class, work on Exercises 1–4, then skip to Exercise 6.** Exercise 6 uses
the original model, so you can do it before homework Exercises 5 and 7. You may start before
class. Use the tutorial to discuss choices, errors, and results with your tutor.

**How to work.** Cells marked **Supplied** are ready to run. Complete **Your turn** cells and the Markdown answer cells. `NotImplementedError` marks missing student work; dependent cells will not work until you complete it. Run cells with **Shift+Enter** and save regularly.

The tutorial times are a working guide. If you need longer, aim first for one complete model, training run, and prediction; finish the remaining code and explanations after class.


## Before class: a short preparation guide

Allow about **20–30 minutes** to check the environment, run the supplied setup
and data cells, and inspect the small examples. You do not need to master a
whole PyTorch course before starting.

1. Copy the **whole released `lab_02` folder** to your personal `my-work/lab_02`
   folder. Keep `lab_02_support.py` beside your notebook. With Colab, upload
   this notebook **and `lab_02_support.py`** into the runtime working folder;
   save your completed notebook outside the temporary runtime.
2. Check the package instructions below. This lab adds **torchvision** for MNIST
   and uses Pillow for the optional image loader.
   Keep a working university or Colab environment. There is no need to install
   CUDA or replace its PyTorch installation merely because a GPU is unavailable.
3. Run the setup and dataset cells below while connected to the internet. The
   first run downloads MNIST into `data/`; later runs reuse it. Downloads may
   disappear when a Colab runtime is reset. Check this before class.
4. Review tensor indexing and `.shape` from Lab 1. Run the short layer example
   in Exercise 2 and the score example in Exercise 3. Bring your questions.


**Package check and installation.** On **MIN or Colab**, try the import cell first.
If `torchvision` or the drawing widget fails on MIN, follow the targeted repair
steps below. Keep the existing PyTorch installation. In Colab, use its existing
PyTorch/torchvision pair and the optional-widget instructions below.
On **Nix**, update the course files and re-enter `nix develop` from the course root;
the provided environment remains CPU-only.

For **local Python**, activate your existing course environment. Run these commands
in its terminal from the **student repository root**, not in a notebook code cell:

```bash
python -m pip install "torch>=2.2,<3" "torchvision>=0.17,<1" --index-url https://download.pytorch.org/whl/cpu
python -m pip install -r lab_02/requirements.txt
```

On **macOS**, omit `--index-url https://download.pytorch.org/whl/cpu` from the first
command. The requirements include `Pillow>=9.1` and Matplotlib. On Windows, use the
course environment's Python executable as shown in `setup/local.md`. Restart your
notebook kernel after installation. Use a compatible PyTorch/torchvision pair;
a `torchvision` import error can indicate incompatible versions.

**Optional drawing pad.** Exercise 6 includes a mouse/touch drawing pad with
the extra package `anywidget` (listed in `requirements-drawing.txt`). The cell
checks this only when you try the pad; it is not needed for the core lab.
Supplied digit images and loading your own PNG/JPEG are available alternatives.

| Project step | Pay attention to | Reference to consult when needed |
|---|---|---|
| Read an image | A batch has shape `(B, C, H, W)`; for MNIST `C=1` | [Tensor basics](https://docs.pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html) |
| Compose layers | Each layer's output must fit the next layer's input | [Build a network: `Flatten`, `Linear`, `ReLU`, `Sequential`](https://docs.pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html) |
| Apply a filter | `in_channels`, `out_channels`, `kernel_size`, `padding` | [`nn.Conv2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html) |
| Reduce spatial size | Pool each channel separately | [`nn.MaxPool2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html) |
| Predict and measure error | Largest score; labels are integers, not ten-entry vectors | [`argmax`](https://docs.pytorch.org/docs/stable/generated/torch.argmax.html), [`CrossEntropyLoss`](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) |
| Evaluate a model | Evaluation mode and calculations without gradients | [`Module.eval`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.eval), [`no_grad`](https://docs.pytorch.org/docs/stable/generated/torch.no_grad.html) |
| Add bounded random noise | Sample tensors of the same shape and clip pixel values | [`torch.rand_like`](https://docs.pytorch.org/docs/stable/generated/torch.rand_like.html), [`torch.clamp`](https://docs.pytorch.org/docs/stable/generated/torch.clamp.html) |

**Understand now:** images, layer composition, shapes, class scores, loss,
validation, and your experimental choices. **Supplied for use:** downloading,
batching, device transfers in the trainer, automatic differentiation, and
parameter updates. Lab 3 will explain the training machinery in detail.
You can read the documented functions in `lab_02_support.py` whenever useful.

Write explanations in Markdown cells. Use `$...$` for inline mathematics;
place `$$` on separate lines around a displayed formula. A notebook with
code, outputs, and written answers is the complete submission; no PDF is needed.


### If a library or the drawing pad does not load

**1. Missing `lab_02_support`.** Keep `lab_02_support.py` in the same folder as this notebook. In Colab, upload it again after a new runtime starts.

**2. Missing or incompatible `torchvision` on MIN.** First run this in a new notebook cell, before the lab's import cell:

```python
import sys
import torch
print(sys.executable)
print(torch.__version__)
```

If PyTorch reports **`2.12.0+cu130`**, use the following command in a separate notebook cell. This is the repair that worked in the instructor's MIN environment:

```python
%pip install --no-deps torchvision==0.27.0 --index-url https://download.pytorch.org/whl/cu130
```

`--no-deps` leaves the existing PyTorch installation in place. Save the notebook, choose **Kernel → Restart Kernel**, and rerun the lab's import cell. If both libraries import successfully, continue with the lab. If your PyTorch version is different, or this repair fails, send the printed version and full error to the tutor so the matching torchvision build can be selected. Use the local setup instructions for local Python; on Nix, update the course files and re-enter `nix develop`.

**3. Missing `anywidget` or `Failed to load model class 'AnyModel' from module 'anywidget'`.** On **MIN, local Python, or Colab**, install the optional drawing package in a new notebook cell:

```python
%pip install "anywidget>=0.9.18,<1"
```

On **MIN**, save all open notebooks, then use **File → Hub Control Panel → Stop My Server → Start My Server**. Reopen JupyterLab and hard-refresh the browser page (`Ctrl+Shift+R`, or `Cmd+Shift+R` on macOS). Restarting only the kernel may not load the newly installed browser extension.

With **local JupyterLab**, stop and relaunch the full JupyterLab server from your course environment, then refresh the browser. In **Colab**, restart the session and rerun the supplied drawing cell, which enables the custom widget manager. On **Nix**, use the supplied environment instead of pip.

A restart clears the trained model from memory: rerun your completed setup, model, and training cells before trying the drawing pad. If the canvas still fails after these steps, send the error to the tutor and use Exercise 6's PNG/JPEG option. The rest of the lab does not need the widget.

Only run a repair when the corresponding import or widget fails. More help and official references are in the course file `setup/troubleshooting.md`.


In [ ]:
# Supplied: imports and device selection.
from pathlib import Path
import torch
import torchvision
import PIL
import matplotlib
from torch import nn
import matplotlib.pyplot as plt
from lab_02_support import (
    make_loaders, train_model, evaluate_model, predict_images,
    plot_history, show_images, count_parameters, load_digit, make_example_digits,
)

SEED = 2026
EPOCHS = 3
LEARNING_RATE = 0.001
BATCH_SIZE = 128
# Keep the CPU workload modest, including on shared university servers.
torch.set_num_threads(min(2, torch.get_num_threads()))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
print("PyTorch:", torch.__version__, "| torchvision:", torchvision.__version__)
print("Pillow:", PIL.__version__, "| Matplotlib:", matplotlib.__version__)
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(device))
else:
    print("CPU is sufficient for the small project; training may take longer.")


## Supplied data: three distinct roles

We use a fixed random subset of **10,000 training images** and a disjoint subset
of **2,000 validation images** from MNIST's official training split. The other
images in that split are unused in this lab. MNIST's **10,000 official test images**
remain separate.

- **Training data:** used to change the network parameters.
- **Validation data:** used to inspect progress and compare the two architectures.
- **Test data:** used once for the final chosen model, at the end of Exercise 5.

A data loader supplies one batch `(images, labels)` at a time. Images are floating-point
tensors in `[0, 1]`; labels are integers from 0 to 9. The first batch has at most
128 images. The last batch can be smaller. The plotting and model code do not
require you to learn NumPy.


In [ ]:
# Supplied. Keep these settings the same for both models in Exercise 5.
train_loader, val_loader, test_loader = make_loaders(
    root="data", batch_size=BATCH_SIZE,
    train_size=10000, val_size=2000, seed=SEED,
)
images, labels = next(iter(train_loader))
print("Training / validation / test sizes:",
      len(train_loader.dataset), len(val_loader.dataset), len(test_loader.dataset))
print("Batch shapes:", images.shape, labels.shape)


## 1. Explore the digit images · Tutorial, 10 minutes

**(a)** Write code that displays at least eight images with their labels, prints the image and label tensor shapes, and reports the minimum and maximum pixel values. Explain every dimension and what a pixel value near zero means.

**(b)** Choose one digit. Make an independent copy and erase a small rectangular patch. Before running your code, state whether you expect the digit to remain recognisable. Display the original and modified image side by side and record your chosen index and patch. Do not alter the training data.

**Useful tools:** tensor indexing, `.shape`, `.min()`, `.max()`, `.item()`, `.clone()`, and `torch.stack`. The supplied `show_images` function takes a batch and optional `labels` and `predictions`.


**Hints.** `images[0]` has shape `(1, 28, 28)`, whereas `images[0, 0]` is its grayscale matrix. You can call `show_images(images, labels=labels, max_images=8)` to display a batch. For a tensor called `picture`, `picture[:, 4:8, 12:16] = 0` erases a rectangle. `torch.stack([first, second])` adds a batch dimension to two individual images. Assemble these operations yourself below.


In [ ]:
# Your turn: inspect and display the data as requested in 1(a).
raise NotImplementedError("Exercise 1(a): write the image inspection code.")


In [ ]:
# Your turn: select a digit, copy it, erase a patch, and display both images.
raise NotImplementedError("Exercise 1(b): write the image modification code.")


**Your answer — 1(a):** Explain shapes and pixel values.

**Your answer — 1(b):** Record your choice, expectation, and observation.


## 2. Construct a CNN · Tutorial, 25 minutes

A layer is a function on tensors. `nn.Sequential(layer1, layer2, ...)` composes
these functions in the listed order. For example, `nn.Linear(3, 2)` maps three
features to two scores for each item in a batch. It includes a learned bias.


In [ ]:
# Supplied small example: first predict the output shape.
tiny_network = nn.Sequential(nn.Linear(3, 2), nn.ReLU())
tiny_input = torch.tensor([[1., 0., -1.], [2., 1., 0.]])
print("Tiny input/output:", tiny_input.shape, tiny_network(tiny_input).shape)

# One convolution: 1 input channel, 4 output channels, a 3-by-3 filter.
# Padding adds a one-pixel border so this stride-one layer keeps H and W.
example_conv = nn.Conv2d(1, 4, kernel_size=3, padding=1)
print("Convolution output:", example_conv(images[:2]).shape)


**Useful tools.**

- `nn.Conv2d(c_in, c_out, kernel_size=3, padding=1)` uses `c_out` learned filters,
  each spanning all `c_in` input channels. Here the stride is one.
- `nn.ReLU()` applies the positive part to every entry.
- `nn.MaxPool2d(2)` takes a maximum over each nonoverlapping $2\times2$ block;
  on our even image sizes it halves both spatial dimensions.
- `nn.Flatten()` keeps the batch dimension and combines all remaining dimensions.
- `nn.Linear(number_of_features, 10)` produces ten **raw class scores** per image.

A convolution with biases has
$$
c_{\mathrm{out}}\bigl(c_{\mathrm{in}} k^2+1\bigr)
$$
trainable parameters for a square filter of width $k$. A linear layer with
$d$ inputs and $m$ outputs has $m(d+1)$ parameters.


**(a)** Write the complete function `make_model()` returning a new `nn.Sequential` network. Translate this architecture from words to code:

1. A convolution from the grayscale input to eight channels, followed by ReLU and max pooling over nonoverlapping 2-by-2 blocks.
2. A convolution from those eight channels to sixteen channels, followed by ReLU and the same pooling operation.
3. Flatten the feature maps and apply an affine layer producing ten raw class scores.

Both convolutions use 3-by-3 filters, stride one, padding one, and biases. The affine layer also has a bias. Do not put ReLU or softmax after the final scores. Use the short examples and layer reference above, but write the complete layer sequence yourself.

**Before running:** list the tensor shape after every layer. Then write a short loop applying the layers one at a time to `images[:2]`, printing the intermediate shapes to check your calculations. A sequential network can be iterated over with `for layer in model:`.

**(b)** Calculate the parameter count of **each trainable layer** by hand, add the counts, and check the total with `count_parameters(model)`. Explain why the convolutional parameter counts do not depend on the image height and width. Identify which layer has the most parameters.

Keep a table of layer names, output shapes, and parameter counts in your answer. The flattening head used in the guided architecture differs from Lecture 2's global-average-pooling head; both are possible design choices.


In [ ]:
# Your turn: write the entire model definition.
def make_model():
    raise NotImplementedError("Exercise 2(a): assemble the layers of your CNN.")


In [ ]:
# Supplied output-contract check, before training.
torch.manual_seed(SEED)
model = make_model()
print(model)
print("Trainable parameters:", count_parameters(model))
assert count_parameters(model) < 50000, "Keep the project below 50,000 parameters."
with torch.no_grad():
    example_scores = model(images[:2])
assert example_scores.shape == (2, 10)
print("Output shape:", example_scores.shape)


In [ ]:
# Your turn: check the intermediate shapes against your written calculations.
raise NotImplementedError("Exercise 2(a): trace the shapes through your model.")


**Your answer — 2(a):** Architecture, reason for each stage, and shape table.

**Your answer — 2(b):** Layer-by-layer parameter calculation, total, and explanation of sharing.


## 3. Train the classifier · Tutorial, 20 minutes

For one image the network produces scores $a_0,\ldots,a_9$. Predict the index
of the largest score. These scores need not be positive and need not sum to one.
The corresponding probabilities and cross-entropy loss for the correct label
$y$ are
$$
p_j=\frac{e^{a_j}}{\sum_{r=0}^{9}e^{a_r}},
\qquad \ell(a,y)=-\log p_y.
$$
`nn.CrossEntropyLoss()` combines these operations in a numerically stable way.
**Pass the raw scores and integer labels directly to it; do not apply softmax first.**
The default result is the mean loss across the batch.

**Useful tools.** `scores.argmax(dim=1)` selects one class per row.
`criterion = nn.CrossEntropyLoss()` creates a loss function;
`criterion(scores, labels)` evaluates it.

**(a)** In the example below, predict both class labels and calculate accuracy
by hand. Then write code to create `criterion`, compute predicted labels, compute the mean cross-entropy loss, and compute the fraction of correct predictions. Print all three results. If all ten scores were zero, what would the
cross-entropy loss be, regardless of the correct label?


In [ ]:
# Supplied example: two images, ten scores per image.
scores = torch.tensor([
    [0., 1., 3., 0., 0., 0., 0., 0., 0., 0.],
    [0., 0., 0., 0., 0., 0., 0., 2., 0., 0.],
])
correct_labels = torch.tensor([2, 1])


In [ ]:
# Your turn: define criterion and compute predictions, mean loss, and accuracy.
raise NotImplementedError("Exercise 3(a): implement the score-to-loss calculation.")


**(b)** Record the untrained model's validation loss and accuracy. Predict what
will happen after training, then run the supplied training routine.

One **epoch** is one pass through all 10,000 training images. Keep the common
budget of three epochs, batch size 128, and learning rate 0.001. The helper
implements Adam and parameter updates; you do not have to reproduce it in this lab.
The validation images are never used for parameter updates.

The plotted training loss averages batches while the parameters are changing.
Validation loss is measured after the epoch with its final parameters. Training
loss can therefore be above validation loss; the two curves measure different
stages of the calculation.

`train_model` moves the model and batches to `device`, trains the model **in place**,
and returns a history for plotting. Running it a second time on the same model
would continue training. For a new experiment, always create a fresh model first.
A seed makes comparisons easier to reproduce on one setup; it does not promise
identical numbers on different CPUs, GPUs, or software versions.


In [ ]:
# Supplied: restart the baseline from the same initialisation before training.
torch.manual_seed(SEED)
model = make_model()
before_training = evaluate_model(model, val_loader, device)
print("Before training:", before_training)
history = train_model(
    model, train_loader, val_loader, criterion=criterion, device=device,
    epochs=EPOCHS, learning_rate=LEARNING_RATE, seed=SEED,
)
after_training = evaluate_model(model, val_loader, device)
print("After training:", after_training)
plot_history(history)


**Your answer — 3(a):** Predictions, accuracy, and equal-score loss.

**Your answer — 3(b):** Your expectation, before/after numbers, and interpretation of the curves. Does one epoch guarantee perfect recognition?


## 4. Evaluate and inspect predictions · Tutorial, 20 minutes

**(a)** Write `predict_labels(model, batch_images, device)`: given one batch of images, return a **CPU tensor** containing one predicted integer label per image. Put the model on `device` and in evaluation mode, use `torch.no_grad()`, move the images to the same device, and choose the largest score in each row.

**(b)** Write `accuracy_on_loader(model, loader, device)`, returning a Python float. Loop over the batches, use your prediction function, count correct predictions and all images, then divide the totals:

$$
\operatorname{accuracy}=\frac{1}{n}\sum_{i=1}^{n}
   \mathbf{1}\{\widehat y_i=y_i\}.
$$

Do not average batch accuracies equally: the final batch may be smaller. Compute validation accuracy and check it against the supplied evaluator. The test set remains reserved for Exercise 5.

**(c)** Write code that selects and displays correct predictions and mistakes from the validation set. Discuss up to two mistakes, if any occur. If there are none in the first batch, search the complete validation set. Handle the case of no mistakes without failing.

**Useful tools:** `model.to(device)`, `model.eval()`, `with torch.no_grad():`, `.to(device)`, `.cpu()`, `.argmax(dim=1)`, `.sum().item()`, and Boolean indexing. Evaluation mode changes the behaviour of layers such as dropout; it does not itself disable gradient recording.


**Programming hints.** Start with the prediction function, test it on `images[:3]`, and only then write the accuracy loop. A function returns its result with `return`. Use two counters, initially zero, and add the number of correct labels and the current batch length at each step.

This tiny example illustrates comparison and selection without solving the evaluation loop:

```python
guesses = torch.tensor([2, 7, 4])
targets = torch.tensor([2, 1, 4])
mask = guesses == targets
print(mask.sum().item())   # Number of matches.
print(guesses[~mask])      # Select the entries where mask is false.
```

The loader gives CPU labels. Returning CPU predictions lets you compare them directly. For the image gallery, collect one prediction tensor per batch in a list, then join the list with `torch.cat`. Use the resulting Boolean masks to select matching images, labels, and predictions together.


In [ ]:
# Your turn: return one CPU label per image, without training the model.
def predict_labels(model, batch_images, device):
    raise NotImplementedError("Exercise 4(a): write batch prediction.")


In [ ]:
# Supplied small interface check.
check_labels = predict_labels(model, images[:3], device)
assert check_labels.shape == (3,) and check_labels.device.type == "cpu"
assert check_labels.dtype == torch.int64
print("Three predicted labels:", check_labels.tolist())


In [ ]:
# Your turn: count correct predictions across the entire loader.
def accuracy_on_loader(model, loader, device):
    raise NotImplementedError("Exercise 4(b): write dataset accuracy.")


In [ ]:
# Supplied check on held-out validation data, not the test set.
validation_accuracy = accuracy_on_loader(model, val_loader, device)
reference_accuracy = evaluate_model(model, val_loader, device)["accuracy"]
print(f"Your validation accuracy: {validation_accuracy:.2%}")
assert abs(validation_accuracy - reference_accuracy) < 1e-10


In [ ]:
# Supplied: collect the small validation set for inspection on the CPU.
image_batches, label_batches = [], []
for batch_images, batch_labels in val_loader:
    image_batches.append(batch_images)
    label_batches.append(batch_labels)
validation_images = torch.cat(image_batches)
validation_labels = torch.cat(label_batches)


In [ ]:
# Your turn: obtain predictions in validation order, select correct and wrong
# predictions, and display examples. Store validation_predictions for later use.
raise NotImplementedError("Exercise 4(c): implement selection and the image galleries.")


**Your answer — 4(a–b):** Report validation accuracy and explain evaluation mode, gradient recording, and why batch size matters in the accuracy calculation.

**Your answer — 4(c):** Discuss up to two mistakes or report that none occurred. Explain why the test set is still reserved.


## 5. Design a second architecture · Homework

**During the tutorial, skip to Exercise 6.** Return here after class. Write your own complete second CNN, rather than changing a single number in the first model.

**(a)** Describe your design and a hypothesis before training. Make **at least one structural change**: change the number of convolutional blocks, the pooling strategy, or the classifier head. Merely increasing a channel count is not enough for this task. You may change more than one feature if the design needs it, but then explain why an observed difference cannot be attributed to one feature alone.

Implement `make_changed_model()` returning a fresh model. Give its layer-by-layer shapes and parameter counts, explain how the layers fit together, and verify your calculations with code. Keep both models below 50,000 trainable parameters and produce ten raw scores per image.

**(b)** Train your new model with the supplied routine. Keep the same data split, three epochs, batch size, learning rate, optimizer, and shuffle seed as the baseline. Compare parameter count, validation loss, validation accuracy, and at least two validation examples on which either model makes a mistake, if available; explain if fewer exist. Explain whether the results support your hypothesis. A design that does not improve accuracy is still a valid result.

**(c)** Write code choosing the model with higher validation accuracy (break a tie by the smaller parameter count). Store it as `selected_model` and its name as `selected_name`. Record your choice before evaluating **that model once** on the official test set with your `accuracy_on_loader` function. Do not use the test result to tune or replace the selected model. A final clean notebook run repeats this same fixed procedure; it is not another design experiment.

**(d)** Connect to Lecture 3: distinguish the existence of parameters with small approximation error, finding useful parameters through training, and predicting new data accurately. Why does one of these not guarantee the others?


**Possible directions — choose one, or propose your own.**

- Keep the convolutional blocks and design a classifier with a hidden affine layer and ReLU before the ten output scores. Work out the flattened input dimension and choose a modest hidden width.
- Compare max pooling with average pooling, or replace the flattening head by global average pooling and an affine classifier. Explain what spatial information is removed.
- Add a convolutional block. Check the spatial dimensions before adding another pooling layer and adjust the classifier input accordingly.

**Hints.** `nn.Linear(d, h)` followed by `nn.ReLU()` and `nn.Linear(h, 10)` creates a nonlinear classifier head. `nn.AvgPool2d(2)` averages local blocks; `nn.AdaptiveAvgPool2d((1, 1))` averages each entire feature map to one value. You do not need all these layers in one network. Consult the [pooling-layer reference](https://docs.pytorch.org/docs/stable/nn.html#pooling-layers) if you choose a new pooling operation.

Write the full new architecture below. Do not modify the trained baseline in place. In part (b), use Boolean masks as in Exercise 4; you can select images where `(baseline_predictions != labels) | (changed_predictions != labels)`.


In [ ]:
# Your turn: design and write the whole second network.
def make_changed_model():
    raise NotImplementedError("Exercise 5(a): implement your second architecture.")


In [ ]:
# Supplied check: a fresh model, not a modification of the trained baseline.
torch.manual_seed(SEED)
changed_model = make_changed_model()
assert count_parameters(changed_model) < 50000
with torch.no_grad():
    assert changed_model(images[:2]).shape == (2, 10)
print(changed_model)
print("Changed-model parameters:", count_parameters(changed_model))


In [ ]:
# Your turn: verify the intermediate shapes and parameter counts of your design.
raise NotImplementedError("Exercise 5(a): check your architecture calculation.")


**Your answer — 5(a), before training:** Describe and justify the structural change, state your hypothesis, give the shape/parameter table, and identify the controlled training conditions.


In [ ]:
# Supplied: the same training budget and the same shuffled data order.
changed_history = train_model(
    changed_model, train_loader, val_loader, criterion=criterion, device=device,
    epochs=EPOCHS, learning_rate=LEARNING_RATE, seed=SEED,
)
baseline_result = evaluate_model(model, val_loader, device)
changed_result = evaluate_model(changed_model, val_loader, device)
print("Baseline:", count_parameters(model), "parameters;", baseline_result)
print("Changed:", count_parameters(changed_model), "parameters;", changed_result)
plot_history(changed_history)


In [ ]:
# Your turn: compare both models on at least two validation images where
# either model makes a mistake (or explain if fewer such images exist).
raise NotImplementedError("Exercise 5(b): compare the models' errors.")


**Your answer — 5(b):** Complete the table, discuss two error examples, and interpret the comparison. Does the experiment support your hypothesis, and what does it leave uncertain?

| Model | Parameters | Validation loss | Validation accuracy |
|---|---:|---:|---:|
| Baseline | | | |
| Your design | | | |

**Your answer — 5(c), before testing:** Record your chosen model and reason.


In [ ]:
# Your turn: use validation accuracy and the parameter-count tie-break only.
# Define selected_model and selected_name, then print your choice.
raise NotImplementedError("Exercise 5(c): implement the selection rule.")


In [ ]:
# Your turn: evaluate the chosen model ONCE with accuracy_on_loader.
# Store final_test_accuracy and report it. Keep your chosen architecture fixed.
raise NotImplementedError("Exercise 5(c): report final test accuracy.")


**Your answer — 5(c), after testing:** Report and interpret test accuracy.

**Your answer — 5(d):** Explain the distinction between representation, training, and generalisation.


## 6. Recognise new digits · Tutorial, 10 minutes

The supplied cell sets `experiment_model` to your selected model if Exercise 5 is complete, and otherwise to the original model trained in Exercise 3. Thus you can do this exercise before homework. Use this model for each image experiment below; do not use these few examples to claim a population accuracy.

**(a)** The supplied code draws three simple **synthetic, handwritten-style**
digits: 0, 1, and 7. These are not taken from MNIST. Inspect the images, predict
which might be difficult, and obtain the model's labels.

**(b)** Modify one of these images: move the digit a little, erase a stroke, or
change its brightness. State your expected effect before making the change,
then compare the predictions. A wrong prediction is a useful result to explain.
Do not assume that convolution and pooling guarantee exact translation invariance.

**Optional own image.** Draw one dark digit on a plain light background and save
a tightly framed PNG or JPEG beside this notebook, for example `my_digit.png`.
An image editor is sufficient; a camera is not required. `load_digit(path,
invert=True)` converts it to grayscale, inverts dark-on-light writing, crops
around the foreground, fits the digit into a $20\times20$ region, and centres
it on a dark $28\times28$ canvas. Inspect this prepared image before interpreting
the prediction. Use `invert=False` for a light digit on a dark background.
This simple preparation does not correct arbitrary shadows or complicated backgrounds.


In [ ]:
# Supplied: examples are generated locally, with no additional download.
new_images = make_example_digits()
intended_labels = torch.tensor([0, 1, 7])
show_images(new_images, labels=intended_labels, max_images=3)

experiment_model = globals().get("selected_model", model)


In [ ]:
# Your turn: use predict_labels with experiment_model and display all three labels.
raise NotImplementedError("Exercise 6(a): predict the supplied new digits.")


**Hint for (b):** start from `new_images.clone()`. To change brightness,
multiply an image by a factor between 0 and 1. To erase a patch, use the
indexing from Exercise 1. Keep the resulting values in `[0, 1]`.


In [ ]:
# Your turn: select one digit, modify a copy, and show before/after predictions.
# Write your expectation in the answer cell before running the experiment.
raise NotImplementedError("Exercise 6(b): implement and inspect your modification.")


### Optional: draw your own digit here

Run the supplied cell, then draw **one digit from 0 to 9** with your mouse or
touchscreen. Click **Recognise** to see the model's prediction and the actual
$28\times28$ image presented to the network. Click **Clear** and try another digit.
You do not need to write any drawing or JavaScript code.

The pad uses your selected model from Exercise 5 if you have completed that
exercise; otherwise it uses the original model from Exercise 3. Run the training
cells first. These few drawings are an informal experiment, not an accuracy estimate.

The drawing pad is **optional**. If `anywidget` is missing, run
`%pip install "anywidget>=0.9.18,<1"` in a new notebook cell on MIN, local Python,
or Colab. For the `AnyModel` error on MIN, save your notebooks and use
**File → Hub Control Panel → Stop My Server → Start My Server**, then reopen
JupyterLab and hard-refresh the browser. A kernel restart alone may not load
the extension. The troubleshooting section near the start of this notebook
gives the complete steps for each setup route. Rerun your model and training
cells after a restart, then try the pad again. If it still fails, use the
image-file option below and send the error to your tutor.

**Save a drawing as ordinary notebook output:** after clicking Recognise, run
the **Record drawing** cell below. This preserves the prepared image and its
prediction when you save the notebook; the interactive pad alone may not survive
a kernel restart or a downloaded copy.


In [ ]:
# Supplied optional drawing pad. Rerunning this cell replaces the previous pad.
previous_pad = globals().get("drawing_pad")
if previous_pad is not None:
    previous_pad.close()
drawing_pad = None

try:
    # Colab needs its custom-widget manager; other environments skip this step.
    try:
        from google.colab import output
        output.enable_custom_widget_manager()
    except ImportError:
        pass
    from IPython.display import display
    from lab_02_support import digit_drawer

    drawing_model = globals().get("selected_model", model)
    drawing_pad = digit_drawer(drawing_model, device)
    display(drawing_pad)
except ImportError as error:
    print("The optional drawing pad is unavailable:", error)
    print('MIN, local Python, or Colab: run %pip install "anywidget>=0.9.18,<1" '
          'in a separate cell, then follow the restart steps near the notebook start.')
    print("MIN: save all notebooks, then File > Hub Control Panel > Stop My Server > "
          "Start My Server; reopen and refresh JupyterLab.")
    print("Nix: re-enter the supplied environment instead of using pip.")
    print("After restarting, rerun the model and training cells before trying the pad.")
    print("You can still use the supplied digits or load an image file below.")


In [ ]:
# Supplied: run AFTER clicking Recognise to save an ordinary image output.
if (drawing_pad is not None and drawing_pad.last_image is not None
        and drawing_pad.last_prediction is not None):
    show_images(drawing_pad.last_image.unsqueeze(0),
                predictions=[drawing_pad.last_prediction], max_images=1)
else:
    print("Draw a digit, click Recognise, then run this cell again.")
    print("If the pad is unavailable, continue with the other image examples.")


### Optional alternative: load an image file

You can use this option without the drawing pad. Save a single digit image
beside the notebook and set its filename below. The image preparation is
described at the beginning of Exercise 6.


In [ ]:
# Supplied optional extension: change None to a filename only if you want to try it.
own_image_path = None  # For example: "my_digit.png"
if own_image_path is not None:
    own_image = load_digit(Path(own_image_path), invert=True)
    own_batch = own_image.unsqueeze(0)  # Add the batch dimension: (1, 1, 28, 28).
    image_model = globals().get("selected_model", model)
    own_prediction = predict_labels(image_model, own_batch, device)
    show_images(own_batch, predictions=own_prediction, max_images=1)


**Your answer — 6(a):** Your expectations and the model's predictions.

**Your answer — 6(b), before running:** Describe the modification and predict its effect.

**Your answer — 6(b), after running:** What happened? Explain why success on MNIST
does not guarantee success on a new style of handwriting or image preparation.

**Optional drawing observation:** If you tried the pad, state one intended digit
and its predicted label. Run the Record drawing cell to keep its image output.


## 7. Test stability under random noise · Homework

Does a small change in the input change the prediction? Use **ten fixed validation images**. Do not discard an image if its clean prediction is wrong. Use `selected_model` from Exercise 5 and keep its parameters fixed. This is a diagnostic experiment, not another model-selection step: do not retrain or switch the model after looking at these results.

**(a)** Write `perturb_images(batch_images, epsilon)`. For each image $x$, return

$$
\widetilde{x}=\operatorname{clip}(x+\varepsilon z,0,1),
\qquad z_{c,h,w}\sim\operatorname{Uniform}[-1,1],
$$

independently for every pixel, image, and repetition. Clipping keeps pixel values in $[0,1]$. Preserve the shape and the original images. Explain why the absolute change in any pixel is at most $\varepsilon$, even after clipping. The clipped noise is generally no longer uniform near the boundaries zero and one.

**(b)** Record the predictions for the clean images. Test noise levels $\varepsilon=0,0.02,0.05,0.10,0.20$, using **ten independently generated perturbations of every image at each nonzero level**. Use the same ten original images at every level. Include all chosen images, without selecting only successful or failed perturbations.

For each level, report two proportions over the 100 image–perturbation pairs:

- **Prediction stability:** how often the perturbed prediction equals that image's clean prediction.
- **Accuracy:** how often the perturbed prediction equals that image's true label.

Write the repeated-experiment code and produce a table or plot. At zero noise, stability should be one and accuracy should equal the clean accuracy; use this as a check. Keep the sample fixed, so changes across levels are not due to choosing different digits.

**(c)** Display at least three original images and corresponding noisy images at two nonzero noise levels, with their predictions. Discuss which images appear sensitive, whether you can still recognise the digits, and why an unchanged prediction need not be correct. If no prediction changes, report that observation; a change is not required. Do the observed proportions have to decrease monotonically with noise level? Explain what this small experiment can and cannot establish about robustness.

**Optional extension:** compare both architectures using exactly the same perturbed images. Do not change the already chosen final model or use the test set for this investigation.


**Hints.** `torch.rand_like(image)` samples independent values in `[0, 1)` with the same shape as `image`. The following example generates one noisy image; adapt it into your function:

```python
noise = epsilon * (2 * torch.rand_like(image) - 1)
noisy_image = (image + noise).clamp(0, 1)
```

Your function can apply the same operations to a batch. For each noise level, use a loop such as `for repetition in range(10):`. Call your function afresh inside the loop, obtain predictions with `predict_labels`, and accumulate two counts: matches with the clean predictions, and matches with the true labels. Divide each count by `10 * len(noise_images)`. Store one result per noise level in a list and display it as a table or plot. Save one repetition's images for the image comparison; do not search for a favourable repetition.


In [ ]:
# Supplied protocol: the first ten validation images, with no filtering by outcome.
noise_images = validation_images[:10].clone()
noise_labels = validation_labels[:10].clone()
noise_levels = [0.0, 0.02, 0.05, 0.10, 0.20]
repetitions = 10
noise_seed = SEED + 7


In [ ]:
# Your turn: preserve shape and input data; keep output pixels in [0, 1].
def perturb_images(batch_images, epsilon):
    raise NotImplementedError("Exercise 7(a): implement bounded random perturbations.")


In [ ]:
# Supplied sanity checks before the repeated experiment.
original_noise_images = noise_images.clone()
noisy_check = perturb_images(noise_images, 0.1)
assert noisy_check.shape == noise_images.shape
assert noisy_check.min() >= 0 and noisy_check.max() <= 1
assert (noisy_check - noise_images).abs().max() <= 0.1 + 1e-6
assert torch.equal(noise_images, original_noise_images), "Do not change the original images."
assert torch.equal(perturb_images(noise_images, 0.0), noise_images)


In [ ]:
# Your turn: seed the experiment, record clean predictions, run all noise levels,
# and report prediction stability and accuracy. Retain examples for part (c).
raise NotImplementedError("Exercise 7(b): implement the repeated-noise experiment.")


In [ ]:
# Your turn: display at least three originals and their noisy versions at two
# nonzero levels, with predictions, using a fixed repetition for illustration.
raise NotImplementedError("Exercise 7(c): display and compare perturbed images.")


**Your answer — 7(a):** Why does clipping preserve the bound on each pixel change?

**Your answer — 7(b):** Present and interpret the stability/accuracy results, including the zero-noise check.

**Your answer — 7(c):** Discuss the displayed images, the distinction between stability and correctness, possible nonmonotonic results, and the limits of this experiment.


## Before submitting

Complete Exercises 1–7 in **one chosen version**, including code, displayed
results, and the Markdown answers. Include your group name and members. After
completing all tasks, restart the kernel and run the notebook from top to bottom;
check that your results and explanations agree, then save the `.ipynb` file.
If you used the drawing pad, draw and recognise a digit again after the clean
run, then run Record drawing before saving your final copy.
Follow Moodle for the deadline and upload location. Keep a backup of your work.
Include your own architecture, its shape and parameter calculations, the controlled comparison, and the noise-stability experiment.
Optional extensions are welcome but are not required. No particular accuracy
threshold is required: a reasonable attempt and a clear explanation of failures
are more useful than an unexplained score. Feedback concerns the shared core
tasks in either version.

**Optional connection to AlexNet.** The operations you used also occur in larger
image-recognition networks. You may compare your architecture with the AlexNet
slide from Lecture 2 or the [Torchvision AlexNet documentation](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.alexnet.html).
This lab does not require downloading AlexNet weights or training AlexNet.
MNIST digit accuracy and ImageNet accuracy concern different classification tasks.
